In [0]:
from pyspark.sql.functions import current_timestamp

# Raw data location
source_path = "/Volumes/automotive_warranty/automotive_service/automotive_raw"

# Create Bronze schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS automotive_warranty.bronze
""")

print("Bronze schema created successfully!")

Bronze schema created successfully!


In [0]:
from pyspark.sql.functions import current_timestamp
import re

# Source: uploaded CSV files
source_path = "/Volumes/automotive_warranty/automotive_service/automotive_raw"

# Bronze schema
bronze_schema = "automotive_warranty.bronze"

# List all files in the raw volume
files = dbutils.fs.ls(source_path)

for file in files:

    # Process only CSV files
    if file.name.lower().endswith(".csv"):

        # Get filename without .csv
        table_name = file.name[:-4]

        # Make sure table name is valid
        table_name = re.sub(r"[^a-zA-Z0-9_]", "_", table_name)

        print(f"Processing: {file.name}")

        # Read CSV
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file.path)
        )

        # Add ingestion timestamp
        df_bronze = df.withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )

        # Write as Delta Bronze table
        (
            df_bronze.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{bronze_schema}.{table_name}")
        )

        print(f"Created Bronze table: {bronze_schema}.{table_name}")

print("========================================")
print("BRONZE INGESTION COMPLETED SUCCESSFULLY!")
print("========================================")

Processing: addresses.csv
Created Bronze table: automotive_warranty.bronze.addresses
Processing: customer_vehicles.csv
Created Bronze table: automotive_warranty.bronze.customer_vehicles
Processing: customers.csv
Created Bronze table: automotive_warranty.bronze.customers
Processing: dealers.csv
Created Bronze table: automotive_warranty.bronze.dealers
Processing: parts_catalog.csv
Created Bronze table: automotive_warranty.bronze.parts_catalog
Processing: parts_used.csv
Created Bronze table: automotive_warranty.bronze.parts_used
Processing: service_appointments.csv
Created Bronze table: automotive_warranty.bronze.service_appointments
Processing: service_centers.csv
Created Bronze table: automotive_warranty.bronze.service_centers
Processing: service_order_tasks.csv
Created Bronze table: automotive_warranty.bronze.service_order_tasks
Processing: service_orders.csv
Created Bronze table: automotive_warranty.bronze.service_orders
Processing: service_types.csv
Created Bronze table: automotive_w

In [0]:
# Verify Bronze tables

spark.sql("SHOW TABLES IN automotive_warranty.bronze").show(50, truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|bronze  |addresses           |false      |
|bronze  |customer_vehicles   |false      |
|bronze  |customers           |false      |
|bronze  |dealers             |false      |
|bronze  |parts_catalog       |false      |
|bronze  |parts_used          |false      |
|bronze  |service_appointments|false      |
|bronze  |service_centers     |false      |
|bronze  |service_order_tasks |false      |
|bronze  |service_orders      |false      |
|bronze  |service_types       |false      |
|bronze  |technicians         |false      |
|bronze  |vehicle_models      |false      |
|bronze  |vehicles            |false      |
|bronze  |warranty_claims     |false      |
+--------+--------------------+-----------+



In [0]:
tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 70)
print("BRONZE TABLE ROW COUNT VALIDATION")
print("=" * 70)

for table in tables:
    count = spark.table(f"automotive_warranty.bronze.{table}").count()
    print(f"{table:<30} : {count:,} rows")

BRONZE TABLE ROW COUNT VALIDATION
addresses                      : 10,100 rows
customer_vehicles              : 10,000 rows
customers                      : 10,000 rows
dealers                        : 20 rows
parts_catalog                  : 50 rows
parts_used                     : 500,000 rows
service_appointments           : 10,000 rows
service_centers                : 50 rows
service_order_tasks            : 500,000 rows
service_orders                 : 10,000 rows
service_types                  : 6 rows
technicians                    : 200 rows
vehicle_models                 : 10 rows
vehicles                       : 10,000 rows
warranty_claims                : 3,500 rows


In [0]:
tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("BRONZE TABLE SCHEMA VALIDATION")
print("=" * 80)

for table in tables:
    print("\n" + "-" * 80)
    print(f"TABLE: automotive_warranty.bronze.{table}")
    print("-" * 80)

    df = spark.table(f"automotive_warranty.bronze.{table}")
    df.printSchema()

print("\n" + "=" * 80)
print("SCHEMA VALIDATION COMPLETED")
print("=" * 80)

BRONZE TABLE SCHEMA VALIDATION

--------------------------------------------------------------------------------
TABLE: automotive_warranty.bronze.addresses
--------------------------------------------------------------------------------
root
 |-- address_id: integer (nullable = true)
 |-- street_address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)


--------------------------------------------------------------------------------
TABLE: automotive_warranty.bronze.customer_vehicles
--------------------------------------------------------------------------------
root
 |-- ownership_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- registration_plate: string (nullable = true)
 |-- purchase_dealer_id: integer (nullable = true)
 |--

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("BRONZE NULL VALUE AUDIT")
print("=" * 80)

for table in tables:
    df = spark.table(f"automotive_warranty.bronze.{table}")

    null_counts = df.select([
        spark_sum(
            col(c).isNull().cast("int")
        ).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    total_nulls = sum(
        value for value in null_counts.values()
        if value is not None
    )

    print(f"{table:<30} : {total_nulls:,} NULL values")

print("=" * 80)
print("NULL VALUE AUDIT COMPLETED")
print("=" * 80)

BRONZE NULL VALUE AUDIT
addresses                      : 0 NULL values
customer_vehicles              : 0 NULL values
customers                      : 0 NULL values
dealers                        : 0 NULL values
parts_catalog                  : 0 NULL values
parts_used                     : 0 NULL values
service_appointments           : 0 NULL values
service_centers                : 0 NULL values
service_order_tasks            : 0 NULL values
service_orders                 : 0 NULL values
service_types                  : 0 NULL values
technicians                    : 0 NULL values
vehicle_models                 : 0 NULL values
vehicles                       : 0 NULL values
warranty_claims                : 0 NULL values
NULL VALUE AUDIT COMPLETED


In [0]:
from pyspark.sql.functions import count

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("BRONZE DUPLICATE AUDIT")
print("=" * 80)

for table in tables:
    df = spark.table(f"automotive_warranty.bronze.{table}")

    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows

    print(f"{table:<30} : {duplicate_rows:,} duplicate rows")

print("=" * 80)
print("DUPLICATE AUDIT COMPLETED")
print("=" * 80)

BRONZE DUPLICATE AUDIT
addresses                      : 0 duplicate rows
customer_vehicles              : 0 duplicate rows
customers                      : 0 duplicate rows
dealers                        : 0 duplicate rows
parts_catalog                  : 0 duplicate rows
parts_used                     : 0 duplicate rows
service_appointments           : 0 duplicate rows
service_centers                : 0 duplicate rows
service_order_tasks            : 0 duplicate rows
service_orders                 : 0 duplicate rows
service_types                  : 0 duplicate rows
technicians                    : 0 duplicate rows
vehicle_models                 : 0 duplicate rows
vehicles                       : 0 duplicate rows
warranty_claims                : 0 duplicate rows
DUPLICATE AUDIT COMPLETED


In [0]:
from pyspark.sql.functions import col, count, when

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("BRONZE DATA QUALITY SUMMARY")
print("=" * 80)

for table in tables:
    df = spark.table(f"automotive_warranty.bronze.{table}")

    total_rows = df.count()

    # NULL count
    null_count = (
        df.select([
            count(when(col(c).isNull(), c)).alias(c)
            for c in df.columns
        ])
        .first()
    )

    total_nulls = sum(null_count)

    # Duplicate count
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows

    # Ingestion timestamp check
    timestamp_nulls = df.filter(
        col("ingestion_timestamp").isNull()
    ).count()

    print(f"\nTable: {table}")
    print(f"  Total rows              : {total_rows:,}")
    print(f"  NULL values             : {total_nulls:,}")
    print(f"  Duplicate rows          : {duplicate_rows:,}")
    print(f"  Missing ingestion time  : {timestamp_nulls:,}")

print("\n" + "=" * 80)
print("BRONZE DATA QUALITY VALIDATION COMPLETED")
print("=" * 80)

BRONZE DATA QUALITY SUMMARY

Table: addresses
  Total rows              : 10,100
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: customer_vehicles
  Total rows              : 10,000
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: customers
  Total rows              : 10,000
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: dealers
  Total rows              : 20
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: parts_catalog
  Total rows              : 50
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: parts_used
  Total rows              : 500,000
  NULL values             : 0
  Duplicate rows          : 0
  Missing ingestion time  : 0

Table: service_appointments
  Total rows              : 10,000
  NULL values             : 0
  Duplicate rows